# Session 14 · XGBoost, AutoML and the Deep-Learning Question

**Data Mining 2026/27** · doc. dr. Uroš Godnov · Mon 11 Jan / Wed 13 Jan 2027 · 3 × 45 min + 15 min break

In S13 a well-tuned logistic regression and a regularised tree reached a ROC-AUC of about 0.76 on the bank-marketing data. Today: **gradient boosting** (the *Gradient Boosted Trees* of AI Studio, in its most famous implementation), a decision threshold chosen by **money**, explanations with **SHAP** — and the question every data scientist now faces: **can AutoML do my job?**

| Part | Time | What we do |
|---|---|---|
| Warm-up | 5 min | install AutoGluon · 🔁 recall S13 · will boosting beat 0.76? |
| **A — Guided walkthrough** | 80 min | leakage check · random forest vs XGBoost · thresholds & lift · SHAP · AutoGluon · deep learning on tables? |
| ☕ Break | 15 min | |
| **B — Your turn** | 35 min | Adult income: XGBoost, a profit-maximising threshold, AutoGluon, your verdict |
| Wrap-up | 10 min | key takeaways, cheat sheet, exit ticket, projects & assignment |
| 🚀 **C — Advanced track** (optional) | after the wrap-up or at home | profit curve in one pass · Optuna · monotonic constraints · a PyTorch network with embeddings · fairness slices · model card |

**Learning objectives** — after this session you can:
1. explain bagging vs boosting, and train a random forest and XGBoost with early stopping;
2. handle class imbalance and choose a decision threshold from business costs (cut-off and lift, now in code);
3. explain a model with permutation importance and SHAP;
4. run AutoGluon, read its leaderboard, and judge what AutoML does — and does not do — for you;
5. argue when deep learning is, and is not, the right tool for tabular data.

| Icon | Meaning |
|---|---|
| 🔮 | **Predict before you run** |
| 🧪 | **Try it** — 2–5 minutes during the walkthrough |
| ✏️ | **TODO** — your task in Part B |
| ✅ | **Self-check** (⏳ = not attempted yet) |
| 🛟 | **Rescue cell** — run only if stuck |
| 🚀 | **Fast lane** — optional side quests in Part A and the Part C advanced track; never graded |
| 📈 💻 | **Two lenses** — a one-minute link to what you already know: 📈 economics and statistics, 💻 programming and databases |
| 🤖 | **AI rule** — try first; then Gemini may *explain* or *hint* |

> 💾 **File → Save a copy in Drive** before you start.

**Data today** (from the web): Bank Marketing (Moro, Cortez & Rita, 2014) · Adult income (Becker & Kohavi, 1996).

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/urosgodnov/dm-2627-python/blob/main/notebooks/s14-xgboost-automl.ipynb)

### 🧭 Two lanes — pick yours

| Lane | For whom | How to work through this notebook |
|---|---|---|
| 🟢 **Core** | everyone — especially if Python is still new to you | Part A with the lecturer → Part B → wrap-up. This is everything the assignment and the project need. |
| 🚀 **Fast lane** | you already write Python comfortably (functions, loops, a bit of pandas) — e.g. you passed the 🚀 check in S00 | Follow Part A, and while we cover basics you know, work on the 🚀 **side quests**. Do Part B quickly (it is still the core of the project), then go on to **Part C · 🚀 Advanced track** after the wrap-up. |

You can switch lanes at any time. Everything marked 🚀 is optional, never graded, and its solutions are in the solutions notebook. Use the 📑 table of contents (left sidebar) to jump between parts.

**Two lenses.** 📈 boxes link the topic to economics and statistics, 💻 boxes to programming and databases. Read the one for your programme, or both. In Part C, 📈 challenges need more statistics and less code; 💻 challenges need more programming. **Strong in statistics but new to Python?** Stay in the 🟢 core for the code, and after the wrap-up try the 🧠 think questions and the 📈 challenges.

## 0 · Warm-up (5 min)

⏳ **Run the next cell first.** It installs AutoGluon, which is not preinstalled in Colab. The installation takes 1–2 minutes — answer the recall questions below while it runs.

In [ ]:
import sys

_pandas_was_loaded = "pandas" in sys.modules
%pip install -q autogluon.tabular
if _pandas_was_loaded:
    print("⚠️ pandas was imported before the installation: use Runtime → Restart session, then run the notebook from the top.")
else:
    print("✔ AutoGluon installed.")

### 🔁 Recall from last session (3 min)

Answer **from memory** first — no scrolling back, no AI. Then run the cell below and compare.

1. A learning curve shows a training AUC of 0.99 and a validation AUC of 0.70. Underfitting or overfitting — and what would you try first?
2. Why do we choose hyperparameters with cross-validation on the training data, and look at the test set only once, at the end?
3. On the bank data, a dummy model that always says “no” reached about 88 % accuracy. Why is that worthless, and which metric did we use instead?

In [ ]:
#@title 👀 Show the answers { display-mode: "form" }
print('1. Overfitting (high variance): the model memorises the training data. Simplify or regularise it (shallower tree, larger leaves, stronger penalty) or add data. Today, early stopping does this job for boosting.')
print('2. Every decision taken while looking at the test set fits the model to that test set, so its score becomes optimistic. The test set stays unseen until the final evaluation — today it also stays away from early stopping.')
print('3. 88 % of the clients said no, so “always no” is 88 % accurate without finding a single subscriber. ROC-AUC measures how well a model ranks the yes-clients above the no-clients (dummy: 0.5). Today we go one step further and turn the ranking into decisions with a threshold chosen by money.')

> 🔮 **Predict before you run:** In S13 the best logistic regression and tuned tree reached a CV ROC-AUC of about **0.76** on the bank data. Will boosting beat it — and by how much?

## ⚙️ Setup

In [ ]:
import io
import warnings
import zipfile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import seaborn as sns
import shap
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import recall_score, precision_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from xgboost import XGBClassifier

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 40)

In [ ]:
#@title 🧰 Course helpers — run this cell (no need to read it) { display-mode: "form" }
CHECK_RESULTS = {}  # label -> True (correct) / False (not yet) / None (not attempted)

def check(label, test, hint=""):
    """Give instant feedback on an exercise without ever stopping 'Run all'.

    `test` is a function (usually a lambda) that returns True when the answer is right.
    """
    try:
        ok = bool(test())
    except NameError as err:
        CHECK_RESULTS[label] = None
        print(f"⏳ {label}: not attempted yet ({err}).")
        return
    except Exception as err:  # wrong type, wrong shape, missing column, ...
        CHECK_RESULTS[label] = False
        print(f"❌ {label}: checking your answer raised {type(err).__name__}: {err}")
        if hint:
            print(f"   💡 Hint: {hint}")
        return
    CHECK_RESULTS[label] = ok
    if ok:
        print(f"✅ {label}: correct!")
    else:
        print(f"❌ {label}: not yet.")
        if hint:
            print(f"   💡 Hint: {hint}")

def progress(*, return_count=False):
    """Summarise all self-checks run so far; optional 🚀 fast-lane checks are counted separately."""
    core = [v for label, v in CHECK_RESULTS.items() if not label.startswith("🚀")]
    fast = [v for label, v in CHECK_RESULTS.items() if label.startswith("🚀")]
    solved = sum(1 for v in core if v)
    print(f"Solved {solved} of {len(core)} exercises checked so far.")
    if fast:
        print(f"🚀 Fast lane (optional): solved {sum(1 for v in fast if v)} of {len(fast)}.")
    for label, v in CHECK_RESULTS.items():
        print(f"  {'✅' if v else ('⏳' if v is None else '❌')} {label}")
    if return_count:
        return solved

print("Helpers loaded ✔")

---
# Part A · Guided walkthrough — bank telemarketing

Same data as S13 Part B: 45,211 calls, 11.7 % said yes to a term deposit. Same split (80/20, stratified, `random_state=42`), so all results are directly comparable.

In [ ]:
BANK_URL = "https://archive.ics.uci.edu/static/public/222/bank+marketing.zip"
outer = zipfile.ZipFile(io.BytesIO(requests.get(BANK_URL, timeout=60).content))
inner = zipfile.ZipFile(io.BytesIO(outer.read("bank.zip")))
bank = pd.read_csv(inner.open("bank-full.csv"), sep=";")
print(bank.shape)

## A1 · One last leakage check: `duration` (5 min)

> 🔮 **Predict before you run:** How well does **one single column**, the call `duration`, separate subscribers from non-subscribers (ROC-AUC)?

In [ ]:
print("ROC-AUC of `duration` alone:", round(roc_auc_score(bank["y"] == "yes", bank["duration"]), 3))

**So what?** One column that is known only *after* the call ranks clients as well as a full model. That is the fingerprint of **leakage** (S13): brilliant in testing, useless in practice, because nobody knows the length of a call before making it. We drop `duration` and keep a **validation** split (`X_val`) inside the training data for early stopping.

In [ ]:
X = bank.drop(columns=["y", "duration"])
y = (bank["y"] == "yes").astype(int)
CAT = list(X.select_dtypes(exclude="number").columns)
NUM = list(X.select_dtypes("number").columns)
X[CAT] = X[CAT].astype("category")          # XGBoost and AutoGluon can use pandas categories directly

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
X_fit, X_val, y_fit, y_val = train_test_split(X_train, y_train, test_size=0.2, stratify=y_train, random_state=42)
print("train", X_train.shape, "→ fit", X_fit.shape, "+ validation", X_val.shape, "| test", X_test.shape)

## A2 · From one tree to many: random forest vs XGBoost (20 min)

| | **Bagging — Random Forest** | **Boosting — XGBoost** |
|---|---|---|
| trees | many **deep** trees, each on a bootstrap sample with random features | many **shallow** trees, built **one after another** |
| idea | average many noisy experts → **variance ↓** | each tree fixes the errors of the previous ones → **bias ↓** |
| risk | little overfitting, rarely needs tuning | overfits if it runs too long → **early stopping** |
| AI Studio | Random Forest | Gradient Boosted Trees |

A *bootstrap sample* draws as many rows as the data has, **with replacement**: some rows appear twice, about a third not at all.

In [ ]:
logreg = Pipeline([("prep", ColumnTransformer([("num", StandardScaler(), NUM),
                                               ("cat", OneHotEncoder(handle_unknown="ignore"), CAT)])),
                   ("clf", LogisticRegression(max_iter=1000))]).fit(X_train, y_train)
p_lr = logreg.predict_proba(X_test)[:, 1]

forest = Pipeline([("prep", ColumnTransformer([("num", "passthrough", NUM),
                                               ("cat", OrdinalEncoder(handle_unknown="use_encoded_value",
                                                                      unknown_value=-1), CAT)])),
                   ("clf", RandomForestClassifier(n_estimators=300, min_samples_leaf=5, n_jobs=-1,
                                                  random_state=42))]).fit(X_train, y_train)
p_rf = forest.predict_proba(X_test)[:, 1]

print(f"logistic regression  test ROC-AUC {roc_auc_score(y_test, p_lr):.3f}")
print(f"random forest        test ROC-AUC {roc_auc_score(y_test, p_rf):.3f}")

The forest beats logistic regression without any tuning: trees find non-linear effects and interactions on their own, which a linear model only learns if we build them by hand.

**XGBoost's key hyperparameters**

| Parameter | What it controls | Typical values |
|---|---|---|
| `n_estimators` | number of trees | let **early stopping** decide |
| `learning_rate` | how much each tree may correct | 0.01–0.1 (smaller = more trees, often better) |
| `max_depth` | depth of each tree | 3–8 |
| `subsample`, `colsample_bytree` | share of rows / columns per tree (randomness against overfitting) | 0.6–0.9 |

**Early stopping** watches a validation set and stops adding trees when the score has not improved for `early_stopping_rounds` rounds.

> ⚠️ **Common mistake:** early stopping on the **test** set (`eval_set=[(X_test, y_test)]`). Early stopping *chooses* the number of trees, so the set it watches becomes part of training and its score turns optimistic. Watch a validation split carved out of the training data (`X_val`) and keep the test set for the final score.

In [ ]:
xgb = XGBClassifier(n_estimators=2000, learning_rate=0.05, max_depth=4, subsample=0.8, colsample_bytree=0.8,
                    enable_categorical=True, tree_method="hist", eval_metric="auc",
                    early_stopping_rounds=100, random_state=42)
xgb.fit(X_fit, y_fit, eval_set=[(X_fit, y_fit), (X_val, y_val)], verbose=False)
p_xgb = xgb.predict_proba(X_test)[:, 1]
print("trees kept by early stopping:", xgb.best_iteration + 1)
print(f"XGBoost              test ROC-AUC {roc_auc_score(y_test, p_xgb):.3f}")

> 🔮 **Predict before you run:** Does adding more and more trees keep improving the **validation** score?

In [ ]:
history = xgb.evals_result()
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(history["validation_0"]["auc"], label="training")
ax.plot(history["validation_1"]["auc"], label="validation")
ax.axvline(xgb.best_iteration, color="grey", linestyle="--", label="early stopping")
ax.set_xlabel("number of trees")
ax.set_ylabel("ROC-AUC")
ax.set_title("Boosting: training keeps improving, validation levels off")
ax.legend()
plt.show()

**So what?** The validation AUC climbs fast, flattens after about 200 trees and barely moves after that, while the training AUC keeps climbing: the extra trees learn noise, not signal. Early stopping kept the best point — a few hundred trees (220 in our Colab test; the exact number depends on the computer) — and XGBoost, at ≈ 0.81, beats the forest (0.80) and logistic regression (0.77).

In [ ]:
# 🧪 Try it: Train XGBoost with its DEFAULT settings (only enable_categorical=True, tree_method="hist",
#    random_state=42) on X_train, y_train — no early stopping. Store the test ROC-AUC in `auc_default`.
#    Is tuning worth it?
# xgb_default = XGBClassifier(...).fit(X_train, y_train)
# auc_default = ...


In [ ]:
#@title ✅ Check — A2 Try it { display-mode: "form" }
check('A2 Try it', lambda: 0.77 < auc_default < 0.81, hint='roc_auc_score(y_test, xgb_default.predict_proba(X_test)[:, 1])')

In [ ]:
# 🚀 Side quest: Out-of-bag (OOB) score — a free validation set. Each tree of a random forest is trained on a
#    bootstrap sample, which leaves out about 37 % of the rows: that tree's "out-of-bag" rows. Predicting
#    every row only with the trees that did NOT see it gives an honest score without a separate split.
#    Refit the forest with oob_score=True (tip: clone(forest).set_params(clf__oob_score=True)) on X_train,
#    y_train and store the OOB ROC-AUC in `auc_oob`. Careful: `oob_score_` is ACCURACY — use the OOB
#    probabilities in `oob_decision_function_[:, 1]`. Compare it with the test AUC and the training AUC.
# forest_oob = clone(forest).set_params(clf__oob_score=True).fit(X_train, y_train)
# auc_oob = roc_auc_score(y_train, ...)


In [ ]:
#@title ✅ Check — 🚀 A2 side quest { display-mode: "form" }
check('🚀 A2 side quest', lambda: abs(auc_oob - roc_auc_score(y_test, p_rf)) < 0.03, hint='forest_oob.named_steps["clf"].oob_decision_function_[:, 1] has one OOB probability per training row.')

## A3 · Imbalance and thresholds: let money decide (20 min)

A model outputs probabilities; a **threshold** turns them into decisions. The default 0.5 is arbitrary — and with 11.7 % positives it means the model says "yes" very rarely:

In [ ]:
for t in [0.5, 0.2]:
    decision = (p_xgb >= t).astype(int)
    print(f"threshold {t}: calls {decision.sum():5d} | recall {recall_score(y_test, decision):.2f} | precision {precision_score(y_test, decision):.2f}")
spw = (y_fit == 0).sum() / (y_fit == 1).sum()
print(f"scale_pos_weight would be {spw:.2f} (no : yes) — an alternative that re-weights the classes during training")

*Recall* = the share of all subscribers that we call; *precision* = the share of calls that reach a subscriber. A lower threshold buys recall with precision.

> ⚠️ **Common mistake:** using `model.predict(X)` for decisions. It silently applies the 0.5 threshold. Use `model.predict_proba(X)[:, 1]` and choose the threshold yourself.

In [ ]:
# 🧪 Try it: Re-weighting instead of a threshold. Train `xgb_spw`: the same settings as `xgb` plus
#    scale_pos_weight=spw, on X_fit with eval_set=[(X_val, y_val)]. Store its test probabilities in `p_spw`
#    and its test ROC-AUC in `auc_spw`. Compare the mean predicted probability and the AUC with `xgb`.
# xgb_spw = XGBClassifier(..., scale_pos_weight=spw)
# xgb_spw.fit(X_fit, y_fit, eval_set=[(X_val, y_val)], verbose=False)
# p_spw = ...
# auc_spw = ...


In [ ]:
#@title ✅ Check — A3 Try it { display-mode: "form" }
check('A3 Try it', lambda: abs(auc_spw - roc_auc_score(y_test, p_xgb)) < 0.02 and p_spw.mean() > 2 * p_xgb.mean(), hint='Copy the XGBClassifier(...) settings of xgb and add scale_pos_weight=spw.')

Re-weighting inflates the probabilities — they no longer match the real share of subscribers — but the **ranking** of clients, and so the AUC, hardly changes. Either way, the decision still needs a threshold (after re-weighting, the best one simply moves up). Let money choose it.

**Business framing** (illustrative numbers): a call costs **€5** of agent time; a subscription earns **€100**.
Expected profit of a calling strategy = €95 · (subscribers called) − €5 · (non-subscribers called).

> 🔮 **Predict before you run:** Which threshold maximises profit — 0.5, 0.3, 0.1 or something even lower?

In [ ]:
COST, BENEFIT = 5, 100
y_true = y_test.to_numpy()

def profit_at(threshold, proba, y_true):
    call = proba >= threshold
    tp = (call & (y_true == 1)).sum()
    fp = (call & (y_true == 0)).sum()
    return tp * (BENEFIT - COST) - fp * COST

thresholds = np.linspace(0.01, 0.9, 90)
profits = np.array([profit_at(t, p_xgb, y_true) for t in thresholds])
best_t = thresholds[profits.argmax()]
call_everyone = profit_at(0.0, p_xgb, y_true)

plt.figure(figsize=(9, 4))
plt.plot(thresholds, profits / 1000)
plt.axvline(best_t, color="green", linestyle="--", label=f"best threshold {best_t:.2f}")
plt.axhline(call_everyone / 1000, color="grey", linestyle=":", label="call everyone")
plt.xlabel("threshold")
plt.ylabel("profit on the test set (k€)")
plt.legend()
plt.title("Expected campaign profit by decision threshold")
plt.show()
print(f"best threshold {best_t:.2f}: €{profits.max():,.0f}  |  threshold 0.5: €{profit_at(0.5, p_xgb, y_true):,.0f}"
      f"  |  call everyone: €{call_everyone:,.0f}")

**So what?** The threshold is a business decision. Chosen by money (≈ 0.05–0.07), it earns almost three times as much as the 0.5 default (≈ €66k vs €23–24k) and also beats calling everyone (≈ €61k) — while calling only about half to two thirds of the clients. The rule of thumb: **break-even probability = cost ÷ benefit**.

> 📈 **Economist's view.** This is expected-utility maximisation: call while the expected marginal benefit, P(yes) × €100, exceeds the marginal cost of €5. When prices change, the optimal threshold moves with them — it is a decision variable, not a property of the model.

> 💻 **Informatics view.** In a scoring system the threshold is a configuration value, not code: store it in a config file or table, version it and log it with every decision. Monitor the score distribution after go-live — if it drifts, the same threshold suddenly calls far more (or fewer) clients.

**Lift** — the AI Studio lift chart in two lines: sort clients by predicted probability and check how many subscribers you reach by calling only the top of the list.

In [ ]:
order = np.argsort(-p_xgb)
reached = np.cumsum(y_true[order]) / y_true.sum()
gains = pd.DataFrame({"share of clients called": [0.1, 0.2, 0.3, 0.5]})
gains["share of subscribers reached"] = [reached[int(f * len(order)) - 1] for f in gains["share of clients called"]]
gains["lift"] = gains["share of subscribers reached"] / gains["share of clients called"]
gains.round(2)

**So what?** Calling the top 10 % of the list reaches about 45 % of all subscribers — a **lift** of about 4.5: at the top of the list the model is four to five times better than random calls. Further down the lift shrinks (≈ 3.2 for the top 20 %, ≈ 1.7 for the top half) — exactly the shape of the AI Studio lift chart.

In [ ]:
# 🚀 Side quest: A decile lift table. Cut the test clients into 10 equal groups by predicted probability
#    (decile 1 = the highest 10 %) with pd.qcut — tip: rank first, pd.Series(p_xgb).rank(method="first",
#    ascending=False), so ties cannot break the bins. Build `lift_table` (one row per decile) with the
#    columns clients, subscribers, response_rate, lift (= response rate ÷ overall rate) and
#    cum_share_reached. Where does the lift drop below 1?
# decile = pd.qcut(pd.Series(p_xgb).rank(method="first", ascending=False), 10, labels=range(1, 11))
# lift_table = (pd.DataFrame({"decile": ..., "subscribed": y_true}).groupby(...)...)


In [ ]:
#@title ✅ Check — 🚀 A3 side quest { display-mode: "form" }
check('🚀 A3 side quest', lambda: len(lift_table) == 10 and lift_table["subscribers"].sum() == y_true.sum() and lift_table["lift"].iloc[0] > 3 and abs(lift_table["cum_share_reached"].iloc[-1] - 1) < 1e-9, hint='groupby on the decile labels; lift = response_rate / y_true.mean(); cumsum() of the subscribers.')

## A4 · Explaining the model: permutation importance and SHAP (10 min)

- **Permutation importance:** shuffle one column at a time and see how much the test score drops.
- **SHAP values:** for each prediction, how much did each feature push it up or down? Averaged, they give a global picture. For XGBoost they are measured in **log-odds**, the model's internal score before it becomes a probability.

In [ ]:
perm = permutation_importance(xgb, X_test, y_test, scoring="roc_auc", n_repeats=5, random_state=42)
importance = pd.Series(perm.importances_mean, index=X_test.columns).sort_values()
importance.tail(10).plot(kind="barh", figsize=(7, 4), title="Permutation importance on the test set")
plt.xlabel("mean drop in test ROC-AUC when the column is shuffled")
plt.show()

In [ ]:
explainer = shap.TreeExplainer(xgb)
X_shap = X_test.sample(1000, random_state=42)          # 1,000 clients give a stable picture, fast
shap_values = explainer(X_shap)
shap.plots.bar(shap_values, max_display=10, show=False)
plt.title("SHAP: average impact on the prediction (1,000 test clients)")
plt.show()

How the client was contacted (`contact`: cellular, telephone or *unknown* — the type was not recorded, mostly for calls in May and June), the **month** of the campaign and the **outcome of the previous campaign** (`poutcome`) drive the predictions. These are explanations of the **model**, not of reality: they say what the model relies on, not what *causes* a client to subscribe.

In [ ]:
# 🧪 Try it: Explain ONE prediction. Find the position of the client with the highest predicted score in
#    `shap_values` (i_top = int(shap_values.values.sum(axis=1).argmax())), draw
#    shap.plots.waterfall(shap_values[i_top]), and store the name of the feature that pushed this client
#    UP the most in `top_reason`.
# i_top = int(shap_values.values.sum(axis=1).argmax())
# shap.plots.waterfall(shap_values[i_top])
# top_reason = ...


In [ ]:
#@title ✅ Check — A4 Try it { display-mode: "form" }
check('A4 Try it', lambda: top_reason == pd.Series(shap_values.values[int(shap_values.values.sum(axis=1).argmax())], index=shap_values.feature_names).idxmax(), hint='pd.Series(shap_values.values[i_top], index=shap_values.feature_names).idxmax()')

> 📈 **Economist's view.** A SHAP value is an attribution inside the model, not a causal effect: a feature can get a large SHAP value simply because it stands in for an unobserved cause (🚀 C6 simulates exactly this). "What if we change X?" needs an experiment or a causal design such as difference-in-differences or instrumental variables.

> 💻 **Informatics view.** In production, a waterfall like this becomes the *reason codes* of a decision: in credit scoring a rejected applicant is usually entitled to the main reasons, so the app stores the top contributions next to each score. Explanations then become a data-engineering task — computed at prediction time, logged and tied to the model version.

In [ ]:
# 🚀 Side quest: SHAP dependence: how does AGE move the prediction? Draw
#    shap.plots.scatter(shap_values[:, "age"]) (one dot per client: age vs its SHAP value). Then store the
#    mean SHAP value of `age` for the bands under 30, 30–59 and 60+ in the Series `age_effect`
#    (tip: pd.cut(X_shap["age"], bins=[0, 29, 59, 120], labels=[...]) and groupby). Which clients does the
#    model favour — and could logistic regression with a plain `age` column learn this shape?
# shap.plots.scatter(shap_values[:, "age"])
# age_band = pd.cut(X_shap["age"], bins=[0, 29, 59, 120], labels=["under 30", "30–59", "60+"])
# age_effect = pd.Series(shap_values[:, "age"].values, index=X_shap.index).groupby(...).mean()


In [ ]:
#@title ✅ Check — 🚀 A4 side quest { display-mode: "form" }
def _check_a4():
    claimed = age_effect.to_numpy(dtype=float)                 # ⏳ until you create it
    band = pd.cut(X_shap["age"], bins=[0, 29, 59, 120])
    expected = pd.Series(shap_values[:, "age"].values, index=X_shap.index).groupby(band, observed=True).mean()
    return len(claimed) == 3 and np.allclose(claimed, expected.to_numpy())

check("🚀 A4 side quest", _check_a4, hint='Bands: bins=[0, 29, 59, 120]; group shap_values[:, "age"].values by them and take the mean.')

## A5 · AutoML: AutoGluon vs you (15 min)

**What AutoML automates:** preprocessing, trying many model families, tuning, and combining the best models into an ensemble (AI Studio: *Auto Model*).
**What it does not do for you:** framing the business problem, deciding which columns are legitimate (leakage!), judging data quality, choosing a threshold from costs.

AutoGluon needs one DataFrame that contains the label column, and a time budget. While it works it prints a log of every model it trains; afterwards its **leaderboard** ranks them. One warning in the log is expected: CatBoost is not installed in Colab, so AutoGluon reports an `ImportError`, skips it and carries on with the libraries it has.

In [ ]:
from autogluon.tabular import TabularPredictor

train_df = X_train.assign(y=y_train.values)
test_df = X_test.assign(y=y_test.values)
automl = TabularPredictor(label="y", eval_metric="roc_auc", path="ag_bank").fit(
    train_df, time_limit=120, presets="medium_quality")

In [ ]:
automl.leaderboard(test_df, silent=True)[["model", "score_test", "score_val", "fit_time"]].head(10)

> ⚠️ **Common mistake:** comparing AutoGluon's `score_val` with your own **test** score. `score_val` comes from AutoGluon's internal validation split, which it also used to pick and weight its models — so it is optimistic. Compare test with test: `score_test`, or `automl.evaluate(test_df)`.

> 🔮 **Predict before you run:** Who wins on the test set: your tuned XGBoost or AutoGluon after two minutes?

In [ ]:
comparison = pd.Series({
    "logistic regression": roc_auc_score(y_test, p_lr),
    "random forest": roc_auc_score(y_test, p_rf),
    "XGBoost (tuned, early stopping)": roc_auc_score(y_test, p_xgb),
    "AutoGluon (2 minutes)": automl.evaluate(test_df, silent=True)["roc_auc"],
}).sort_values(ascending=False).round(3)
comparison

**So what?** Four very different model families end within about 0.04 AUC of each other, and two minutes of AutoGluon add at most a few thousandths to a well-set XGBoost. On a clean table the algorithm is rarely the bottleneck — decisions about the data are, as the next cell shows.

**AutoML does not know what is legitimate.** Give it the leaky `duration` column and it happily uses it:

In [ ]:
leaky_train = train_df.assign(duration=bank.loc[X_train.index, "duration"].values)
leaky_test = test_df.assign(duration=bank.loc[X_test.index, "duration"].values)
leaky = TabularPredictor(label="y", eval_metric="roc_auc", path="ag_bank_leaky", verbosity=0).fit(
    leaky_train, time_limit=60, presets="medium_quality")
print("AutoGluon WITH duration — test ROC-AUC:", round(leaky.evaluate(leaky_test, silent=True)["roc_auc"], 3),
      "← impressive, and useless: duration is unknown before the call")

> 📈 **Economist's view.** AutoML optimises prediction, not explanation: it tells you *who* is likely to subscribe, never *why*, or what would happen if the bank changed its offer. For a policy question you still need a model of the mechanism and an identification strategy.

> 💻 **Informatics view.** For engineers, AutoML automates the model-building step — the rest of MLOps remains: data pipelines, versioning, deployment, monitoring and retraining. Keep an AutoML run reproducible (data snapshot, time budget, library versions), so a model in production can be rebuilt and audited.

In [ ]:
# 🚀 Side quest: Does AutoML also win in euros? Take AutoGluon's test probabilities
#    (automl.predict_proba(test_df)[1].to_numpy()), compute the profit for the Part A `thresholds` with
#    `profit_at`, and store the best profit in `profit_ag_best` and its threshold in `t_ag_best`.
#    Compare with XGBoost's best profit (profits.max()). Is the small AUC gain worth anything here?
# p_ag = automl.predict_proba(test_df)[1].to_numpy()
# profits_ag = ...
# t_ag_best, profit_ag_best = ...


In [ ]:
#@title ✅ Check — 🚀 A5 side quest { display-mode: "form" }
def _check_a5():
    claimed = profit_ag_best                                   # ⏳ until you create it
    p = automl.predict_proba(test_df)[1].to_numpy()
    grid = np.linspace(0.01, 0.9, 90)
    best = max(95 * ((p >= t) & (y_true == 1)).sum() - 5 * ((p >= t) & (y_true == 0)).sum() for t in grid)
    return claimed == best

check("🚀 A5 side quest", _check_a5, hint="profit_at(t, p_ag, y_true) for every t in thresholds; then .max() and .argmax().")

## A6 · The deep-learning question (5 min)

Does a neural network beat boosted trees here? A small multi-layer perceptron, trained on the same data:

In [ ]:
mlp = Pipeline([("prep", ColumnTransformer([("num", StandardScaler(), NUM),
                                            ("cat", OneHotEncoder(handle_unknown="ignore"), CAT)])),
                ("clf", MLPClassifier(hidden_layer_sizes=(64, 32), early_stopping=True, max_iter=200,
                                      random_state=42))]).fit(X_train, y_train)
print(f"neural network (MLP)  test ROC-AUC {roc_auc_score(y_test, mlp.predict_proba(X_test)[:, 1]):.3f}")
print(f"XGBoost               test ROC-AUC {roc_auc_score(y_test, p_xgb):.3f}")

**Discussion.** On **tabular** data, tree ensembles usually beat neural networks while being faster, easier to tune and easier to explain (Grinsztajn, Oyallon & Varoquaux, 2022). AutoGluon agrees: `NeuralNetFastAI` takes a large share of the two-minute budget (≈ 50 s in Colab) and still ranks below every boosted-tree model in the leaderboard above, and `NeuralNetTorch` is usually skipped for lack of time — look for "Skipping NeuralNetTorch" in its log. Deep learning shines on **unstructured** data — text, images, audio — like the **embeddings** you used with JJ for hotel reviews. *Tabular foundation models* (e.g. TabPFN) are starting to change the picture for small tables — see *Going further*. (🚀 Part C4 gives the network a fairer chance, with embeddings for the categories.)

> **Before the break:** read the Part B scenario below.

---
## ☕ Break — 15 minutes

---
# Part B · Your turn: a premium card for high earners (35 min)

**Scenario.** Back to the S11 data: the bank wants to mail a premium-card offer to people with an income above 50K. Each offer costs **€10**; an offer that reaches a high earner brings an expected margin of **€50**. You explored these data in S11 — now you build the model, choose the threshold and challenge AutoML.

In [ ]:
ADULT_URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data"
ADULT_COLUMNS = ["age", "workclass", "fnlwgt", "education", "education_num", "marital_status",
                 "occupation", "relationship", "race", "sex", "capital_gain", "capital_loss",
                 "hours_per_week", "native_country", "income"]
adult = pd.read_csv(ADULT_URL, header=None, names=ADULT_COLUMNS, skipinitialspace=True, na_values="?")

Xa = adult.drop(columns="income")
ya = (adult["income"] == ">50K").astype(int)
CAT_A = list(Xa.select_dtypes(exclude="number").columns)
Xa[CAT_A] = Xa[CAT_A].astype("category")
Xa_train, Xa_test, ya_train, ya_test = train_test_split(Xa, ya, test_size=0.2, stratify=ya, random_state=42)
Xa_fit, Xa_val, ya_fit, ya_val = train_test_split(Xa_train, ya_train, test_size=0.2, stratify=ya_train, random_state=42)
print("train", Xa_train.shape, "| test", Xa_test.shape, "| share of high earners", round(ya.mean(), 3))

In [ ]:
# ✏️ TODO: Task 1 — XGBoost with early stopping. Train `xgb_adult` exactly like `xgb` in Part A (same
#    hyperparameters, enable_categorical=True, early_stopping_rounds=100) on Xa_fit / ya_fit with
#    eval_set=[(Xa_val, ya_val)]. Store the test probabilities in `p_adult` and the test ROC-AUC in `auc_xgb_adult`.
# xgb_adult = XGBClassifier(...)
# xgb_adult.fit(Xa_fit, ya_fit, eval_set=[(Xa_val, ya_val)], verbose=False)
# p_adult = ...
# auc_xgb_adult = ...


In [ ]:
#@title ✅ Check — Task 1 { display-mode: "form" }
check('Task 1', lambda: auc_xgb_adult > 0.92, hint='Copy the XGBClassifier(...) settings from A2; predict_proba(Xa_test)[:, 1].')

In [ ]:
#@title 🛟 Rescue — run only if you could not finish Task 1 { display-mode: "form" }
xgb_adult = XGBClassifier(n_estimators=2000, learning_rate=0.05, max_depth=4, subsample=0.8,
                          colsample_bytree=0.8, enable_categorical=True, tree_method="hist",
                          eval_metric="auc", early_stopping_rounds=100, random_state=42)
xgb_adult.fit(Xa_fit, ya_fit, eval_set=[(Xa_val, ya_val)], verbose=False)
p_adult = xgb_adult.predict_proba(Xa_test)[:, 1]
print("xgb_adult trained, p_adult ready.")

In [ ]:
# ✏️ TODO: Task 2 — The profit-maximising threshold. With an offer cost of €10 and a margin of €50 per high
#    earner reached, compute the profit on the test set for thresholds np.linspace(0.01, 0.9, 90).
#    Store the best threshold in `best_threshold_adult`, its profit in `best_profit_adult` and the profit at
#    threshold 0.5 in `profit_05_adult`. (Tip: reuse `profit_at` from A3 after adapting COST and BENEFIT.)
# COST, BENEFIT = 10, 50
# profits_adult = np.array([profit_at(t, p_adult, ya_test.to_numpy()) for t in thresholds])
# best_threshold_adult = ...
# best_profit_adult = ...
# profit_05_adult = ...


In [ ]:
#@title ✅ Check — Task 2 { display-mode: "form" }
check('Task 2', lambda: 0.1 <= best_threshold_adult <= 0.3 and best_profit_adult > profit_05_adult, hint='profit_at uses the global COST and BENEFIT; argmax() gives the position of the best threshold.')

**✍️ Your answer** — Why does the best threshold land near 0.2, far below 0.5? Write the rule in one sentence.

> *(double-click this cell and write your answer here)*

In [ ]:
# ✏️ TODO: Task 3 — Challenge AutoML. Train a TabularPredictor (label="income_high", eval_metric="roc_auc",
#    path="ag_adult", verbosity=0) on the training data for 60 seconds (time_limit=60, presets="medium_quality").
#    Store the fitted predictor in `automl_adult` and its test ROC-AUC in `auc_automl_adult`.
# automl_adult = TabularPredictor(...).fit(Xa_train.assign(income_high=ya_train.values), time_limit=60, ...)
# auc_automl_adult = automl_adult.evaluate(Xa_test.assign(income_high=ya_test.values), silent=True)["roc_auc"]


In [ ]:
#@title ✅ Check — Task 3 { display-mode: "form" }
check('Task 3', lambda: auc_automl_adult > 0.90, hint='The label column must be part of the DataFrame you pass to fit(): Xa_train.assign(income_high=...).')

**✍️ Your answer** — **Task 4 — Your verdict (5 sentences): human vs AutoML.** Who won, by how much, and what did *you* contribute that AutoML could not?

> *(double-click this cell and write your answer here)*

In [ ]:
# ✏️ ⭐ Stretch: Explain `xgb_adult` with SHAP: compute SHAP values for 1,000 test rows, draw the bar plot,
#    and store the names of the three features with the largest mean |SHAP| in the list `top3_adult`.
# explainer_adult = shap.TreeExplainer(xgb_adult)
# sv_adult = explainer_adult(Xa_test.sample(1000, random_state=42))
# top3_adult = ...


In [ ]:
#@title ✅ Check — Stretch { display-mode: "form" }
check('Stretch', lambda: len(top3_adult) == 3 and set(top3_adult) <= set(Xa_test.columns), hint='np.abs(sv_adult.values).mean(axis=0) gives one mean |SHAP| per feature.')

> 🚀 **Done early?** First make sure Tasks 1–4 are solid — they are exactly what your project and the assignment need. Then continue with **Part C · 🚀 Advanced track** after the wrap-up (in class if time is left, otherwise at home).

---
## Wrap-up (10 min)

**Key takeaways**
1. Boosting (XGBoost) is the workhorse for tabular data; early stopping stops it before it overfits.
2. Thresholds are business decisions: break-even probability = cost ÷ benefit. Lift shows how much better than random the ranking is.
3. Permutation importance and SHAP explain the **model**, not causality.
4. AutoML is a powerful, fast baseline — but leakage, data quality, fairness and decisions remain your job.
5. Deep learning shines on text, images and audio; on tables, tree ensembles usually win.

In [ ]:
progress()

### 📋 Cheat sheet — today's essentials

| I want to … | Code |
|---|---|
| Random forest (bagging) | `RandomForestClassifier(n_estimators=300, min_samples_leaf=5, n_jobs=-1)` |
| XGBoost with early stopping | `XGBClassifier(n_estimators=2000, learning_rate=0.05, early_stopping_rounds=100, enable_categorical=True, eval_metric="auc")` |
| … watching a validation split (never the test set) | `xgb.fit(X_fit, y_fit, eval_set=[(X_val, y_val)], verbose=False)` |
| How many trees did early stopping keep? | `xgb.best_iteration + 1` |
| Probabilities instead of 0/1 decisions | `proba = model.predict_proba(X_test)[:, 1]` |
| Decide with your own threshold | `call = proba >= threshold` |
| Break-even threshold from business costs | `threshold = cost / benefit` |
| Profit of a decision rule | `tp * (benefit - cost) - fp * cost` |
| Share of positives reached from the top of the list | `np.cumsum(y[np.argsort(-proba)]) / y.sum()` |
| Permutation importance | `permutation_importance(model, X_test, y_test, scoring="roc_auc")` |
| SHAP: global picture and one prediction | `sv = shap.TreeExplainer(model)(X_sample); shap.plots.bar(sv); shap.plots.waterfall(sv[0])` |
| AutoML baseline and its leaderboard | `TabularPredictor(label="y", eval_metric="roc_auc").fit(train_df, time_limit=120).leaderboard(test_df)` |

> 🧩 **For your project:** (1) Start with honest baselines — a dummy model and a logistic regression. (2) Train XGBoost with early stopping on a validation split. (3) Run AutoGluon for a few minutes as the benchmark to beat — it is also the yardstick of the assignment. (4) Hunt for leakage: drop every column that is known only after the outcome. (5) Turn probabilities into a decision with a cost-based threshold, and report the result in euros, not only in AUC. (6) Explain the model with SHAP — and say clearly that it explains the model, not the causes.

**✍️ Your answer** — **Exit ticket.** (1) Bagging or boosting — which reduces variance, which bias? (2) Calling costs €2 and a sale earns €40: what is the break-even probability? (3) Name two things AutoML cannot decide for you.

> *(double-click this cell and write your answer here)*

**That's it for the Python part — well done!** Next: project work (11–15 Jan, consultations on request) and the **project presentations on Mon 18 Jan**. The individual assignment ("Beat the AutoML") uses everything from S10–S14.

## 🧠 Think it through (optional · no code · 10 min)

Questions for thinking rather than coding, e.g. if you are strong in statistics but new to Python. Discuss them in pairs, or write two or three sentences for each. The solutions notebook has model answers.

1. 📈 The bank's cost per call rises from €5 to €10, while a subscription still earns €100. What happens to the best threshold and to the number of calls — and does the model have to be retrained?
2. 📈 A colleague reads the SHAP plot and proposes: "March and October work best — let's make all the calls in March." What is wrong with this reasoning?
3. 💻 Your XGBoost model goes live in the bank's call-centre app. What must you store or log so that, half a year later, you can reproduce and explain one specific decision?
4. 💻 AutoGluon's ensemble of five models beats your single XGBoost by 0.001 AUC. Which one would you deploy, and what would you weigh?

> ✍️ *(double-click this cell and write your answers here)*

---
# Part C · 🚀 Advanced track (optional)

**For:** fast-lane students who finished Part B — or anyone who wants more, at home. Never graded; the assignment and the project do not need it.

**Theme:** Pro tools for gradient boosting — the questions a senior data scientist asks before a model goes live: can the decision be computed exactly, can the tuning be smarter, does the model behave sensibly, could a neural network do better, who pays for its errors — and does its explanation tell us what causes what?

| # | Challenge | Time | Level | Lens |
|---|---|---|---|---|
| **C1** | A profit curve in one pass — sort + cumulative sums | 10 min | 🌶️🌶️ | 💻 |
| **C2** | Tune with Optuna — a smarter search with a time budget | 15 min | 🌶️🌶️ | 💻 |
| **C3** | Monotonic constraints — more education never lowers the score | 15 min | 🌶️🌶️ | 📈 |
| **C4** | A neural network that respects categories — PyTorch + entity embeddings | 25 min | 🌶️🌶️🌶️ | 💻 |
| **C5** | Who pays for the errors? — fairness slices | 15 min | 🌶️🌶️ | 📈 |
| **C6** | An explanation is not a cause — SHAP vs the true effect in a simulation | 15 min | 🌶️🌶️ | 📈 |
| **🏁** | Open: a one-page model card | 20 min | open | 📈💻 |

**Lens:** 📈 more statistics and economics, less code · 💻 more programming and data engineering · 📈💻 both

Each challenge stands on its own. **Run the ⚙️ Part C setup cell first**, then pick any challenge. The usual rules apply: a first own attempt, then Gemini may *explain* or give a *hint*.

C2 installs **Optuna** (≈ 15 s) and deliberately searches for one minute. C4 trains a small PyTorch network on the CPU — no GPU needed. Every solution runs in under 90 seconds.

In [ ]:
#@title ⚙️ Part C setup — run this first (rebuilds everything Part C needs) { display-mode: "form" }
if "check" not in globals():  # runtime restarted? then restore the ✅ self-check helpers
    CHECK_RESULTS = {}  # label -> True (correct) / False (not yet) / None (not attempted)

    def check(label, test, hint=""):
        """Give instant feedback on an exercise without ever stopping 'Run all'.

        `test` is a function (usually a lambda) that returns True when the answer is right.
        """
        try:
            ok = bool(test())
        except NameError as err:
            CHECK_RESULTS[label] = None
            print(f"⏳ {label}: not attempted yet ({err}).")
            return
        except Exception as err:  # wrong type, wrong shape, missing column, ...
            CHECK_RESULTS[label] = False
            print(f"❌ {label}: checking your answer raised {type(err).__name__}: {err}")
            if hint:
                print(f"   💡 Hint: {hint}")
            return
        CHECK_RESULTS[label] = ok
        if ok:
            print(f"✅ {label}: correct!")
        else:
            print(f"❌ {label}: not yet.")
            if hint:
                print(f"   💡 Hint: {hint}")

    def progress(*, return_count=False):
        """Summarise all self-checks run so far; optional 🚀 fast-lane checks are counted separately."""
        core = [v for label, v in CHECK_RESULTS.items() if not label.startswith("🚀")]
        fast = [v for label, v in CHECK_RESULTS.items() if label.startswith("🚀")]
        solved = sum(1 for v in core if v)
        print(f"Solved {solved} of {len(core)} exercises checked so far.")
        if fast:
            print(f"🚀 Fast lane (optional): solved {sum(1 for v in fast if v)} of {len(fast)}.")
        for label, v in CHECK_RESULTS.items():
            print(f"  {'✅' if v else ('⏳' if v is None else '❌')} {label}")
        if return_count:
            return solved

    print("Helpers loaded ✔")

import io
import time
import warnings
import zipfile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import shap
import torch
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch import nn
from xgboost import XGBClassifier

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

if "X_fit" not in globals():  # Part A: bank data and splits
    BANK_URL = "https://archive.ics.uci.edu/static/public/222/bank+marketing.zip"
    outer = zipfile.ZipFile(io.BytesIO(requests.get(BANK_URL, timeout=60).content))
    inner = zipfile.ZipFile(io.BytesIO(outer.read("bank.zip")))
    bank = pd.read_csv(inner.open("bank-full.csv"), sep=";")
    X = bank.drop(columns=["y", "duration"])
    y = (bank["y"] == "yes").astype(int)
    CAT = list(X.select_dtypes(exclude="number").columns)
    NUM = list(X.select_dtypes("number").columns)
    X[CAT] = X[CAT].astype("category")          # XGBoost and AutoGluon can use pandas categories directly

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
    X_fit, X_val, y_fit, y_val = train_test_split(X_train, y_train, test_size=0.2, stratify=y_train, random_state=42)
if "xgb" not in globals():  # Part A2: the tuned XGBoost
    xgb = XGBClassifier(n_estimators=2000, learning_rate=0.05, max_depth=4, subsample=0.8, colsample_bytree=0.8,
                        enable_categorical=True, tree_method="hist", eval_metric="auc",
                        early_stopping_rounds=100, random_state=42)
    xgb.fit(X_fit, y_fit, eval_set=[(X_fit, y_fit), (X_val, y_val)], verbose=False)
    p_xgb = xgb.predict_proba(X_test)[:, 1]
if "profit_at" not in globals():  # Part A3: profit of one threshold
    COST, BENEFIT = 5, 100

    def profit_at(threshold, proba, y_true):
        call = proba >= threshold
        tp = (call & (y_true == 1)).sum()
        fp = (call & (y_true == 0)).sum()
        return tp * (BENEFIT - COST) - fp * COST
if "Xa_fit" not in globals():  # Part B: Adult data and splits
    ADULT_URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data"
    ADULT_COLUMNS = ["age", "workclass", "fnlwgt", "education", "education_num", "marital_status",
                     "occupation", "relationship", "race", "sex", "capital_gain", "capital_loss",
                     "hours_per_week", "native_country", "income"]
    adult = pd.read_csv(ADULT_URL, header=None, names=ADULT_COLUMNS, skipinitialspace=True, na_values="?")

    Xa = adult.drop(columns="income")
    ya = (adult["income"] == ">50K").astype(int)
    CAT_A = list(Xa.select_dtypes(exclude="number").columns)
    Xa[CAT_A] = Xa[CAT_A].astype("category")
    Xa_train, Xa_test, ya_train, ya_test = train_test_split(Xa, ya, test_size=0.2, stratify=ya, random_state=42)
    Xa_fit, Xa_val, ya_fit, ya_val = train_test_split(Xa_train, ya_train, test_size=0.2, stratify=ya_train, random_state=42)
if "xgb_adult" not in globals():  # Part B Task 1 (rebuilt if you skipped it)
    xgb_adult = XGBClassifier(n_estimators=2000, learning_rate=0.05, max_depth=4, subsample=0.8,
                              colsample_bytree=0.8, enable_categorical=True, tree_method="hist",
                              eval_metric="auc", early_stopping_rounds=100, random_state=42)
    xgb_adult.fit(Xa_fit, ya_fit, eval_set=[(Xa_val, ya_val)], verbose=False)
    p_adult = xgb_adult.predict_proba(Xa_test)[:, 1]

p_xgb = xgb.predict_proba(X_test)[:, 1]
y_true = y_test.to_numpy()
p_adult = xgb_adult.predict_proba(Xa_test)[:, 1]
ya_true = ya_test.to_numpy()
t_adult = float(best_threshold_adult) if "best_threshold_adult" in globals() else 10 / 50   # your Part B threshold
print(f"Part C ready ✔  bank XGBoost test AUC {roc_auc_score(y_test, p_xgb):.3f} · "
      f"Adult XGBoost test AUC {roc_auc_score(ya_test, p_adult):.3f} · Adult threshold t_adult = {t_adult:.2f}")

## C1 · A profit curve in one pass 🌶️🌶️ (10 min) · 💻

**Why it matters.** The A3 loop recomputes the whole confusion matrix for every threshold. That is fine for 90 thresholds, but the *exact* optimum is hidden between grid points, and a production system may re-score millions of clients.

**What to build.** `profit_fast(proba, y_true, thresholds, cost, benefit)` that returns the profit for every threshold **without a Python loop**:
1. sort the clients by probability, best first;
2. the gain of each call is `benefit − cost` for a subscriber and `−cost` otherwise; a cumulative sum gives the profit of calling the top *k* clients, for every *k* at once (put a 0 in front for *k* = 0);
3. for each threshold, `np.searchsorted` finds *k* = how many clients have `proba >= threshold`.

Then: (a) store `fast_profits` for the 90 Part A thresholds and confirm it equals the loop; (b) time both on **every distinct probability** (≈ 9,000 thresholds); (c) store the exact best threshold in `best_t_exact` and its profit in `best_profit_exact`.

*Hints:* `np.argsort(-proba)`; `np.searchsorted` needs an ascending array — search in `-proba_sorted` for `-thresholds` with `side="right"`; `time.perf_counter()`. `profit_at` reads the global `COST` and `BENEFIT`, which Part B changed — set them back to 5 and 100 first (and notice why functions should take such numbers as arguments).

In [ ]:
# 🚀 Challenge: Write profit_fast(proba, y_true, thresholds, cost, benefit) — no Python loop. Store
#    `fast_profits` (90 Part A thresholds), time the loop and profit_fast on np.unique(p_xgb), and store
#    `best_t_exact` and `best_profit_exact`.
# COST, BENEFIT = 5, 100
# def profit_fast(proba, y_true, thresholds, cost, benefit):
#     order = np.argsort(-proba, kind="stable")
#     gain = ...
#     cum = np.concatenate([[0], np.cumsum(gain)])
#     k = np.searchsorted(..., side="right")
#     return cum[k]
#
# fast_profits = profit_fast(p_xgb, y_true, np.linspace(0.01, 0.9, 90), 5, 100)
# candidates = np.unique(p_xgb)
# ... time the loop and profit_fast with time.perf_counter() ...
# best_t_exact, best_profit_exact = ...


In [ ]:
#@title ✅ Check — 🚀 C1 { display-mode: "form" }
def _check_c1():
    claimed = np.asarray(fast_profits)                         # ⏳ until you create it
    grid = np.linspace(0.01, 0.9, 90)
    loop = np.array([95 * ((p_xgb >= t) & (y_true == 1)).sum() - 5 * ((p_xgb >= t) & (y_true == 0)).sum() for t in grid])
    order = np.argsort(-p_xgb, kind="stable")
    cum = np.concatenate([[0], np.cumsum(np.where(y_true[order] == 1, 95, -5))])
    exact = cum[np.searchsorted(-p_xgb[order], -np.unique(p_xgb), side="right")].max()
    called = (p_xgb >= best_t_exact).sum()
    return (np.array_equal(claimed, loop) and best_profit_exact == exact
            and cum[called] == exact)

check("🚀 C1", _check_c1, hint="fast_profits must equal the loop exactly; best_profit_exact is the max over np.unique(p_xgb).")

## C2 · Tune with Optuna 🌶️🌶️ (15 min) · 💻

**Why it matters.** The Part A hyperparameters were an educated guess. A grid over six parameters would need thousands of fits; **Optuna**'s TPE sampler learns from earlier trials where the good settings are and samples there more often.

**What to build.**
1. `objective(trial)`: sample `max_depth` (2–8), `learning_rate` (0.05–0.3, log scale — smaller rates need many more trees and would eat the budget), `subsample` and `colsample_bytree` (0.5–1), `min_child_weight` (1–50, log) and `reg_lambda` (0.01–10, log); train XGBoost (at most 1,000 trees, `early_stopping_rounds=50`) with early stopping on `(X_val, y_val)`; return the **validation** AUC (`model.best_score`).
2. A study with `TPESampler(seed=42, n_startup_trials=5)` (5 random trials, then TPE); `study.enqueue_trial(...)` with the Part A settings, so the search starts from them; `study.optimize(objective, n_trials=40, timeout=60)` — a **time budget** of one minute.
3. Evaluate the best trial's model on the test set **once**: `auc_optuna`. (Keep every trial's model in a dict, `models[trial.number] = model`, so nothing has to be refitted.)

*Hints:* `trial.suggest_int("max_depth", 2, 8)`, `trial.suggest_float("learning_rate", 0.05, 0.3, log=True)`; `trial.set_user_attr("trees", model.best_iteration + 1)` records the number of trees; `study.trials_dataframe()` shows every trial. How many trials fit into the minute depends on the computer.

In [ ]:
%pip install -q optuna
import optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)      # no log line per trial
print("Optuna", optuna.__version__)

In [ ]:
# 🚀 Challenge: Tune XGBoost on the bank data with Optuna: objective(trial) → validation AUC, a seeded TPE
#    study that starts from the Part A settings, timeout=60 s; then ONE test evaluation of the best trial's
#    model, stored in `auc_optuna`.
# def objective(trial):
#     params = {"max_depth": trial.suggest_int("max_depth", 2, 8),
#               "learning_rate": trial.suggest_float("learning_rate", 0.05, 0.3, log=True),
#               ...}
#     model = XGBClassifier(n_estimators=1000, early_stopping_rounds=50, eval_metric="auc",
#                           enable_categorical=True, tree_method="hist", random_state=42, **params)
#     model.fit(X_fit, y_fit, eval_set=[(X_val, y_val)], verbose=False)
#     return model.best_score
#
# study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42, n_startup_trials=5))
# study.enqueue_trial({...the Part A settings...})
# study.optimize(objective, n_trials=40, timeout=60)
# auc_optuna = ...   (the best trial's model on X_test, once)


In [ ]:
#@title ✅ Check — 🚀 C2 { display-mode: "form" }
check('🚀 C2', lambda: len(study.trials) >= 1 and study.trials[0].params.get("learning_rate") == 0.05 and study.best_value >= study.trials[0].value and 0.79 < auc_optuna < 0.83, hint="Enqueue the Part A settings as trial 0; return model.best_score (validation AUC) from objective; evaluate the best trial's model on X_test once.")

**✍️ Your answer** — Did Optuna beat the Part A settings on the **test** set — and would a gain of a few thousandths be bigger than the noise?

> *(double-click this cell and write your answer here)*

## C3 · Monotonic constraints 🌶️🌶️ (15 min) · 📈

**Why it matters.** A model that sometimes predicts a *lower* income for *more* education is hard to defend to a manager or a regulator, even if its AUC is fine. XGBoost can be told that a feature may only push the prediction in one direction.

**What to build.**
1. Drop `education`: it is the same information as `education_num`, as a category — and the model could route around the constraint through it.
2. Train two Adult models with the Part B settings: `free` (no constraint) and `mono` with `monotone_constraints={"education_num": 1}`.
3. An **ICE check** (*individual conditional expectation*): take 500 test people, set `education_num` to every value 1–16 in turn and predict. A person is a *violation* if their curve ever goes **down**. Store the counts in `violations_free` and `violations_mono`, and the test AUCs in `auc_free` and `auc_mono`.

*Hints:* `rows.assign(education_num=v)`; `np.column_stack` of 16 prediction vectors; `(np.diff(ice, axis=1) < 0).any(axis=1).sum()`.

In [ ]:
# 🚀 Challenge: Monotonic constraint on education_num (+1): train `free` and `mono` without the
#    `education` column, run the ICE check on 500 test people, and store `violations_free`,
#    `violations_mono`, `auc_free` and `auc_mono`.
# COLS_C3 = [c for c in Xa_fit.columns if c != "education"]
# free = XGBClassifier(...).fit(Xa_fit[COLS_C3], ya_fit, eval_set=[(Xa_val[COLS_C3], ya_val)], verbose=False)
# mono = XGBClassifier(..., monotone_constraints={"education_num": 1}).fit(...)
# rows = Xa_test[COLS_C3].sample(500, random_state=42)
# ice_free = np.column_stack([free.predict_proba(rows.assign(education_num=v))[:, 1] for v in range(1, 17)])
# violations_free = ...
# auc_free, auc_mono = ...


In [ ]:
#@title ✅ Check — 🚀 C3 { display-mode: "form" }
check('🚀 C3', lambda: violations_mono == 0 and violations_free > 0 and auc_mono > 0.92 and abs(auc_free - auc_mono) < 0.01, hint='monotone_constraints={"education_num": 1}; count rows of np.diff(ice, axis=1) < 0 with .any(axis=1).')

## C4 · A neural network that respects categories 🌶️🌶️🌶️ (25 min) · 💻

**Why it matters.** In A6 the scikit-learn MLP saw one-hot columns and lost to XGBoost. An **entity embedding** (Guo & Berkhahn, 2016) gives every category a small learned vector instead — the same idea as word embeddings — and is the standard way to feed tables to a neural network. Does it close the gap?

**What to build** (bank data, the Part A splits):
1. Inputs: the category codes of the 9 categorical columns (`X[c].cat.codes + 1`, so 0 is free for "missing") and the 6 numeric columns, standardised with a scaler fitted on `X_fit` only.
2. A PyTorch module: one `nn.Embedding(n_categories + 1, dim)` per categorical column (dim = min(10, (n + 2) // 2)), concatenated with the numeric columns, then `Linear → ReLU → Dropout → Linear → ReLU → Linear(…, 1)`.
3. Training: `BCEWithLogitsLoss`, Adam (lr = 0.003), mini-batches of 1,024 rows (large batches keep the CPU busy and the loop short), seeds fixed (`torch.manual_seed(42)` and a seeded `torch.Generator` for shuffling). After each epoch compute the **validation AUC**; stop after 5 epochs without improvement and restore the best weights. Keep it under 90 seconds.
4. Store the test AUC in `auc_emb` and the best epoch in `best_epoch`, and compare with XGBoost (`p_xgb`).

*Hints:* `torch.set_num_threads(1)` first — for a network this small, one CPU thread is many times faster than several, which mostly wait for each other; `torch.randperm(n, generator=g).split(1024)` gives shuffled mini-batches; `net.eval()` + `torch.no_grad()` for predictions; `{k: v.clone() for k, v in net.state_dict().items()}` saves the best weights.

In [ ]:
# 🚀 Challenge: PyTorch MLP with entity embeddings on the bank data: early stopping on the validation
#    AUC (patience 5), seeds fixed, < 90 s on the CPU. Store the test ROC-AUC in `auc_emb` and the
#    best epoch in `best_epoch`.
# torch.manual_seed(42)
# torch.set_num_threads(1)
# scaler = StandardScaler().fit(X_fit[NUM])
# def to_tensors(df):
#     codes = np.stack([df[c].cat.codes.to_numpy() + 1 for c in CAT], axis=1)
#     return torch.tensor(codes, dtype=torch.long), torch.tensor(scaler.transform(df[NUM]), dtype=torch.float32)
#
# class EmbeddingMLP(nn.Module):
#     def __init__(self, cardinalities, n_numeric):
#         super().__init__()
#         self.embeddings = nn.ModuleList(...)
#         self.mlp = nn.Sequential(...)
#     def forward(self, x_cat, x_num):
#         ...
#
# ... training loop (Adam, lr=3e-3, batches of 1,024) with early stopping on the validation AUC ...
# auc_emb, best_epoch = ...


In [ ]:
#@title ✅ Check — 🚀 C4 { display-mode: "form" }
check('🚀 C4', lambda: 0.77 < auc_emb < 0.83 and 1 <= best_epoch <= 40, hint='Restore the best weights before predicting X_test; compute the AUC on probabilities or logits (same ranking).')

**✍️ Your answer** — Did the embeddings close the gap to XGBoost? What would you try next — and is it worth it for this table?

> *(double-click this cell and write your answer here)*

## C5 · Who pays for the errors? 🌶️🌶️ (15 min) · 📈

**Why it matters.** One threshold for everyone *sounds* fair, but the model's errors can still fall unevenly on groups. Which error matters depends on the decision: here a false positive wastes €10 of postage, a false negative means a high earner never sees the offer — for a loan or a job, the stakes would be far higher.

**What to build.** `slice_report(group, proba, y_true, threshold)` that returns one row per group with the columns `n`, `base_rate` (share of high earners), `selection_rate` (share who get the offer), `TPR` (share of the high earners who get it), `FPR` (share of the others who get it) and `AUC`. Run it on the Adult test set at your Part B threshold (`t_adult`) for `sex` → `fair_sex`, and for four age bands (17–29, 30–44, 45–59, 60+; `pd.cut`) → `fair_age`. Then write the reflection below.

*Hints:* `df.groupby("group", observed=True)`; TPR = mean of `selected` among the rows with `y == 1`.

In [ ]:
# 🚀 Challenge: Fairness slices of the Adult model at t_adult: slice_report(...) → `fair_sex` (by sex) and
#    `fair_age` (four age bands) with the columns n, base_rate, selection_rate, TPR, FPR, AUC.
# def slice_report(group, proba, y_true, threshold):
#     df = pd.DataFrame({"group": np.asarray(group), "y": y_true, "p": proba, "selected": proba >= threshold})
#     rows = {}
#     for g, d in df.groupby("group", observed=True):
#         rows[g] = {"n": ..., "base_rate": ..., "selection_rate": ..., "TPR": ..., "FPR": ..., "AUC": ...}
#     return pd.DataFrame(rows).T
#
# age_band = pd.cut(Xa_test["age"], bins=[0, 29, 44, 59, 120], labels=["17–29", "30–44", "45–59", "60+"])
# fair_sex = ...
# fair_age = ...


In [ ]:
#@title ✅ Check — 🚀 C5 { display-mode: "form" }
def _check_c5():
    report = fair_sex.loc[["Female", "Male"]]                  # ⏳ until you create it
    sex = Xa_test["sex"].to_numpy()
    selected = p_adult >= t_adult
    for g in ["Female", "Male"]:
        in_g = sex == g
        expected = [selected[in_g].mean(), selected[in_g & (ya_true == 1)].mean(), selected[in_g & (ya_true == 0)].mean()]
        if not np.allclose(report.loc[g, ["selection_rate", "TPR", "FPR"]].astype(float), expected):
            return False
    return len(fair_age) == 4 and {"n", "base_rate", "selection_rate", "TPR", "FPR", "AUC"} <= set(fair_age.columns)

check("🚀 C5", _check_c5, hint="selection_rate = mean of (p_adult >= t_adult) in the group; TPR among y == 1, FPR among y == 0.")

**✍️ Your answer** — **Reflection — who pays for the errors?** Compare the groups. Is the model unfair, the data, or neither? Would deleting the `sex` column fix it? What would change if the same model decided on **loans** instead of mailings?

> *(double-click this cell and write your answer here)*

## C6 · An explanation is not a cause 🌶️🌶️ (15 min) · 📈

**Why it matters.** Managers read a SHAP plot as "push this feature and the outcome moves". With observational data that can be badly wrong, because a hidden **confounder** may drive both the feature and the outcome. In a simulation you know the truth, so you can check.

**What to build.** The next cell simulates 20,000 bank clients. An unobserved `affluence` drives both the use of the bank's mobile app (`app_user`) and subscribing; the app itself has **no effect at all** (its coefficient is 0).
1. Fit XGBoost `xgb_c6` on the observed columns `Xc6` (`app_user`, `age`, `tenure`, `region_code`) and store `app_user`'s share of the total mean |SHAP| in `shap_share_app`.
2. **The model's "effect"**: predict for everyone once with `app_user` = 1 and once with `app_user` = 0, and store the difference of the mean probabilities in `model_effect`.
3. **The true causal effect**: intervene in the simulation — `simulate_y(app_user)` with everyone set to 1, then to 0 (same affluence, age and noise) — and store the difference of the subscription rates in `true_effect`.

*Hints:* `Xc6.assign(app_user=1)`; `np.abs(sv.values).mean(axis=0)` per feature; `simulate_y(np.ones(N_C6, dtype=int))`.

In [ ]:
rng_c6 = np.random.default_rng(42)
N_C6 = 20_000
affluence = rng_c6.normal(size=N_C6)                                # the confounder — the model never sees it
age_c6 = rng_c6.integers(18, 80, size=N_C6)
tenure_c6 = rng_c6.integers(0, 30, size=N_C6)                       # pure noise
region_c6 = rng_c6.integers(0, 5, size=N_C6)                        # pure noise
app_user = (affluence + rng_c6.normal(size=N_C6) > 0.5).astype(int)  # affluent clients use the app more often
noise_c6 = rng_c6.logistic(size=N_C6)

def simulate_y(app_user, affluence=affluence, age=age_c6, noise=noise_c6):
    # The TRUE outcome model: affluence and age matter, the app does NOT (its coefficient is 0).
    score = -2.5 + 1.5 * affluence + 0.0 * app_user + 0.02 * (age - 45)
    return (score + noise > 0).astype(int)

y_c6 = simulate_y(app_user)
Xc6 = pd.DataFrame({"app_user": app_user, "age": age_c6, "tenure": tenure_c6, "region_code": region_c6})
print(f"{N_C6:,} simulated clients | {y_c6.mean():.1%} subscribe | app users: {y_c6[app_user == 1].mean():.1%}, "
      f"others: {y_c6[app_user == 0].mean():.1%}")

In [ ]:
# 🚀 Challenge: Fit `xgb_c6` on Xc6, y_c6; store `shap_share_app` (app_user's share of the mean |SHAP|),
#    `model_effect` (mean predicted P with app_user = 1 minus with app_user = 0) and `true_effect`
#    (the same difference from simulate_y — an intervention in the true model).
# xgb_c6 = XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=3, random_state=42).fit(Xc6, y_c6)
# sv = shap.TreeExplainer(xgb_c6)(Xc6.sample(2000, random_state=42))
# shap_share_app = ...
# model_effect = xgb_c6.predict_proba(Xc6.assign(app_user=1))[:, 1].mean() - ...
# true_effect = simulate_y(np.ones(N_C6, dtype=int)).mean() - ...


In [ ]:
#@title ✅ Check — 🚀 C6 { display-mode: "form" }
def _check_c6():
    share = float(shap_share_app)                              # ⏳ until you create it
    implied = (xgb_c6.predict_proba(Xc6.assign(app_user=1))[:, 1].mean()
               - xgb_c6.predict_proba(Xc6.assign(app_user=0))[:, 1].mean())
    truth = simulate_y(np.ones(len(Xc6), dtype=int)).mean() - simulate_y(np.zeros(len(Xc6), dtype=int)).mean()
    return (0.3 < share <= 1 and abs(model_effect - implied) < 1e-6 and model_effect > 0.05
            and abs(true_effect - truth) < 0.02)

check("🚀 C6", _check_c6, hint="model_effect: the same fitted model on Xc6.assign(app_user=1) vs =0; "
                              "true_effect: simulate_y with everyone set to 1 vs 0.")

**✍️ Your answer** — Why does the model "believe" in the app — and how could the bank estimate the app's real effect from real data?

> *(double-click this cell and write your answer here)*

## 🏁 Open challenge · A one-page model card (20 min) · 📈💻

A **model card** (Mitchell et al., 2019) is the "package leaflet" of a model: what it is for, how well it works, for whom it works less well and what it must not be used for. Write one for the Adult XGBoost — max. one page, with the numbers from this notebook (Part B, C3, C5):

**Model details** · **Intended use** and **out-of-scope uses** · **Training and evaluation data** · **Metrics** (AUC, profit at your threshold) · **Results by group** (C5) · **Limitations and ethical considerations** · **How to monitor it** after deployment.

**✍️ Your answer** — A one-page model card for the Adult XGBoost (use the headings above).

> *(double-click this cell and write your answer here)*

In [ ]:
progress()

---
## Going further (optional, self-study)

- **LightGBM** and **CatBoost** — XGBoost's main rivals (`lightgbm` is preinstalled in Colab).
- **Optuna** (Part C2) — add *pruning* to stop bad trials early; **FLAML** — a lightweight AutoML alternative.
- AutoGluon `presets="best_quality"` — bagging and stacking for the last few tenths of a point (needs much more time).
- **TabPFN** — a pre-trained transformer for small tables (strong up to ~10,000 rows; needs a GPU and accepting a non-commercial licence).
- **Fairlearn** — fairness metrics and mitigation for scikit-learn-style models (Part C5 by hand).
- **Causal inference** (after C6): Hernán & Robins, *Causal Inference: What If* (2020) · Cunningham, *Causal Inference: The Mixtape* (2021) — both free online.
- Read: Grinsztajn, Oyallon & Varoquaux (2022), *Why do tree-based models still outperform deep learning on typical tabular data?* NeurIPS Datasets & Benchmarks · Guo & Berkhahn (2016), *Entity embeddings of categorical variables*, arXiv:1604.06737 · Mitchell et al. (2019), *Model cards for model reporting*, FAT* '19.

**Data sources and licences:** Bank Marketing — Moro, Cortez & Rita (2014), UCI, CC BY 4.0 · Adult — Becker & Kohavi (1996), UCI, CC BY 4.0.